In [ ]:
import logging

import openeo.processes
from utils import urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
# threshold applied to probability of cropland
cropland_probability_threshold = 0.1

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

# Load Uganda ADM-4 boundaries

geoboundaries URL https://www.geoboundaries.org/api/current/gbHumanitarian/UGA/ADM4/

In [ ]:
ADM_BOUNDARIES_URL = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbHumanitarian/UGA/ADM4/geoBoundaries-UGA-ADM4.geojson"

adm_boundaries = connection.load_url(
    ADM_BOUNDARIES_URL,
    format="GeoJSON",
)

In [ ]:
adm_boundaries.metadata.dimension_names()

In [ ]:
process_graph_results.append(
    adm_boundaries.save_result(
        format="GeoJSON",
        options={
            "filename_prefix": "0000_adm_boundaries",
        },
    )
)

In [ ]:
# adm_boundaries = adm_boundaries.filter_bbox(extent=spatial_extent)
# OpenEoApiError: [400] ProcessParameterInvalid: The value passed for parameter 'data' in process 'filter_bbox' is invalid: Expected raster cube but got vector cube.

In [ ]:
vectorcube_filter_bbox_udf = openeo.UDF.from_file(
    "../udf/vectorcube_filter_bbox.py",
    runtime="Python",
    version="3.11",
    # context set here is ignored! 😠
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
adm_boundaries = adm_boundaries.apply_dimension(
    process=vectorcube_filter_bbox_udf,
    dimension="geometry",
    # try passing context here as well 🤷‍♂️
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
process_graph_results.append(
    adm_boundaries.save_result(
        format="GeoJSON",
        options={
            "filename_prefix": "0001_adm_boundaries",
        },
    )
)

# Load decimal year of deforestation

In [ ]:
# load results from previous batch job
JOB_ID = "j-26080615434744c6985547c10c25c56f"

deforestation_year = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    deforestation_year.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0010_deforestation_year",
        },
    )
)

In [ ]:
deforestation_year = deforestation_year.drop_dimension("t")

In [ ]:
process_graph_results.append(
    deforestation_year.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0011_deforestation_year",
        },
    )
)

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-2609111429014f79ba27580bfcfc33eb"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_forest_baseline_mask",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 ⚠️ 😠
# https://forum.dataspace.copernicus.eu/t/round-trip-nodata/5512
forest_baseline_mask = forest_baseline_mask == 1

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0022_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0022_forest_baseline_mask",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0025_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0025_forest_baseline_mask",
        },
    )
)

# load cropland probability

In [ ]:
cropland_probability = connection.load_stac(
    url=urls.MEAN_CROPS_STAC,
    spatial_extent=spatial_extent,
    bands=["crops"],
)

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_cropland_probability",
        },
    )
)

In [ ]:
cropland_probability = utils.drop_hidden_dimension(cropland_probability, "t")

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0031_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0031_cropland_probability",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
cropland_probability = cropland_probability.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0032_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0032_cropland_probability",
        },
    )
)

In [ ]:
cropland_mask = cropland_probability.band("crops") > cropland_probability_threshold

In [ ]:
process_graph_results.append(
    cropland_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_cropland_mask",
        },
    )
)
process_graph_results.append(
    cropland_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_cropland_mask",
        },
    )
)

# Structure of following code

The CDSE openEO backend has basically no support for processing vector data.
Ideally I would generate each KPI, then merge them together into a single output.
However, this is not possible.

https://forum.dataspace.copernicus.eu/t/merge-vector-cubes/5425/

Essentially, `aggregate_spatial` has to be the last process,
because after that the data is a vector cube, and there is no facility to process it any futher. 😡

# KPI 0: Forest stock baseline

In [ ]:
# re-label bands whilst still a raster datacube
forest_stock_baseline = forest_baseline_mask.rename_labels(
    dimension="bands",
    target=["forest_stock_baseline_ha"],
)

In [ ]:
process_graph_results.append(
    forest_stock_baseline.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_forest_stock_baseline",
        },
    )
)
process_graph_results.append(
    forest_stock_baseline.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_forest_stock_baseline",
        },
    )
)

# KPI 1: Forest loss in each year

It would be nice if this wasn't written out for each year manually.
But it does have to be a UDP.

In [ ]:
deforestation_yr_band = deforestation_year.band("data")

In [ ]:
deforestation_2020 = (deforestation_yr_band >= 2020) & (deforestation_yr_band < 2021)
deforestation_2020_datacube = deforestation_2020.add_dimension(
    name="bands", label="deforestation_2020_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    deforestation_2020_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_deforestation_2020",
        },
    )
)

In [ ]:
deforestation_2021 = (deforestation_yr_band >= 2021) & (deforestation_yr_band < 2022)
deforestation_2021_datacube = deforestation_2021.add_dimension(
    name="bands", label="deforestation_2021_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    deforestation_2021_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_deforestation_2021",
        },
    )
)

In [ ]:
deforestation_2022 = (deforestation_yr_band >= 2022) & (deforestation_yr_band < 2023)
deforestation_2022_datacube = deforestation_2022.add_dimension(
    name="bands", label="deforestation_2022_ha", type="bands"
)


In [ ]:
process_graph_results.append(
    deforestation_2022_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_deforestation_2022",
        },
    )
)

In [ ]:
deforestation_2023 = (deforestation_yr_band >= 2023) & (deforestation_yr_band < 2024)
deforestation_2023_datacube = deforestation_2023.add_dimension(
    name="bands", label="deforestation_2023_ha", type="bands"
)


In [ ]:
process_graph_results.append(
    deforestation_2023_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_deforestation_2023",
        },
    )
)

In [ ]:
deforestation_2024 = (deforestation_yr_band >= 2024) & (deforestation_yr_band < 2025)
deforestation_2024_datacube = deforestation_2024.add_dimension(
    name="bands", label="deforestation_2024_ha", type="bands"
)


In [ ]:
process_graph_results.append(
    deforestation_2024_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_deforestation_2024",
        },
    )
)

# KPI 2: Forest loss to cropland

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

band0 = cropland_mask.add_dimension("bands", label="cropland_mask", type="bands")
band1 = deforestation_2020.add_dimension(
    "bands", label="deforestation_2020", type="bands"
)
band2 = deforestation_2021.add_dimension(
    "bands", label="deforestation_2021", type="bands"
)
band3 = deforestation_2022.add_dimension(
    "bands", label="deforestation_2022", type="bands"
)
band4 = deforestation_2023.add_dimension(
    "bands", label="deforestation_2023", type="bands"
)
band5 = deforestation_2024.add_dimension(
    "bands", label="deforestation_2024", type="bands"
)

combined = (
    band0.merge_cubes(band1)
    .merge_cubes(band2)
    .merge_cubes(band3)
    .merge_cubes(band4)
    .merge_cubes(band5)
)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_combined",
        },
    )
)

In [ ]:
forest_loss_to_cropland_2020 = combined.band("cropland_mask") & combined.band(
    "deforestation_2020"
)
forest_loss_to_cropland_2020_datacube = forest_loss_to_cropland_2020.add_dimension(
    name="bands", label="forest_loss_to_cropland_2020_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    forest_loss_to_cropland_2020.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_forest_loss_to_cropland_2020",
        },
    )
)

In [ ]:
forest_loss_to_cropland_2021 = combined.band("cropland_mask") & combined.band(
    "deforestation_2021"
)
forest_loss_to_cropland_2021_datacube = forest_loss_to_cropland_2021.add_dimension(
    name="bands", label="forest_loss_to_cropland_2021_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    forest_loss_to_cropland_2021.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_forest_loss_to_cropland_2021",
        },
    )
)

In [ ]:
forest_loss_to_cropland_2022 = combined.band("cropland_mask") & combined.band(
    "deforestation_2022"
)
forest_loss_to_cropland_2022_datacube = forest_loss_to_cropland_2022.add_dimension(
    name="bands", label="forest_loss_to_cropland_2022_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    forest_loss_to_cropland_2022.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_forest_loss_to_cropland_2022",
        },
    )
)

In [ ]:
forest_loss_to_cropland_2023 = combined.band("cropland_mask") & combined.band(
    "deforestation_2023"
)
forest_loss_to_cropland_2023_datacube = forest_loss_to_cropland_2023.add_dimension(
    name="bands", label="forest_loss_to_cropland_2023_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    forest_loss_to_cropland_2023.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_forest_loss_to_cropland_2023",
        },
    )
)

In [ ]:
forest_loss_to_cropland_2024 = combined.band("cropland_mask") & combined.band(
    "deforestation_2024"
)
forest_loss_to_cropland_2024_datacube = forest_loss_to_cropland_2024.add_dimension(
    name="bands", label="forest_loss_to_cropland_2024_ha", type="bands"
)

In [ ]:
process_graph_results.append(
    forest_loss_to_cropland_2024.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_forest_loss_to_cropland_2024",
        },
    )
)

# Stack all KPI layers

In [ ]:
kpis_raster = (
    # KPI 0
    forest_stock_baseline
    # KPI 1
    .merge_cubes(deforestation_2020_datacube)
    .merge_cubes(deforestation_2021_datacube)
    .merge_cubes(deforestation_2022_datacube)
    .merge_cubes(deforestation_2023_datacube)
    .merge_cubes(deforestation_2024_datacube)
    # KPI 2
    .merge_cubes(forest_loss_to_cropland_2020_datacube)
    .merge_cubes(forest_loss_to_cropland_2021_datacube)
    .merge_cubes(forest_loss_to_cropland_2022_datacube)
    .merge_cubes(forest_loss_to_cropland_2023_datacube)
    .merge_cubes(forest_loss_to_cropland_2024_datacube)
)

In [ ]:
process_graph_results.append(
    kpis_raster.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_kpis_raster",
        },
    )
)
process_graph_results.append(
    kpis_raster.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_kpis_raster",
        },
    )
)

convert pixel masks to pixel area (units: ha)

In [ ]:
# 1 ha = 10000 m^2
kpis_raster = kpis_raster.apply(lambda x: x * resample_spatial_resolution * resample_spatial_resolution / 10000)

In [ ]:
process_graph_results.append(
    kpis_raster.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0081_kpis_raster",
        },
    )
)
process_graph_results.append(
    kpis_raster.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0081_kpis_raster",
        },
    )
)

raster stats

In [ ]:
kpis_vector = kpis_raster.aggregate_spatial(
    geometries=adm_boundaries, reducer=openeo.processes.sum
)

In [ ]:
# json format is pretty useless - no metadata (column names)
# process_graph_results.append(
#     kpis_vector.save_result(
#         format="JSON",
#     )
# )

# CSV format is also pretty useless - only indexed by feature_id
# no geometry or feature properties
# process_graph_results.append(
#     kpis_vector.save_result(
#         format="CSV",
#         options={
#             "filename_prefix": "0090_kpis_vector",
#             # https://forum.dataspace.copernicus.eu/t/access-to-geojson-properties/5418/
#             "feature_id_property": "shapeName",
#         },
#     )
# )

process_graph_results.append(
    kpis_vector.save_result(
        format="Parquet",
        options={
            "filename_prefix": "0090_kpis_vector",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job(
    # https://forum.dataspace.copernicus.eu/t/multiresult-with-geojson/5416
    # Adds a UUID to each output, but multiple outputs overwrite each other 😠
    # job_options={"stac-version": "1.1"}
)
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)